# Loss Functions in TensorFlow

This notebook is the TensorFlow companion to the PyTorch lab. For every loss function implemented in PyTorch, you will find the TensorFlow/Keras equivalent — or implement it from scratch when no built-in exists.

| Concept | PyTorch | TensorFlow/Keras |
|---------|---------|------------------|
| MSE | `nn.MSELoss()` | `tf.keras.losses.MeanSquaredError()` |
| BCE from logits | `nn.BCEWithLogitsLoss()` | `BinaryCrossentropy(from_logits=True)` |
| Categorical CE | `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |
| KL divergence | `nn.KLDivLoss(reduction='batchmean')` | `tf.keras.losses.KLDivergence()` |
| Custom loss | `nn.Module` subclass | `tf.keras.losses.Loss` subclass |

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow version: {tf.__version__}')

def check(name, a, b, atol=1e-5):
    a_np = a.numpy() if hasattr(a, 'numpy') else np.array(a)
    b_np = b.numpy() if hasattr(b, 'numpy') else np.array(b)
    ok = np.allclose(a_np, b_np, atol=atol)
    status = '✓' if ok else '✗'
    print(f'{status} {name}: max_diff={np.abs(a_np - b_np).max():.2e}')

## Section 1 — Regression Losses: MSE, MAE, Huber

In [ ]:
y_pred = tf.constant([ 1.5, -0.5,  2.0,  0.0,  3.0], dtype=tf.float32)
y_true = tf.constant([ 1.0,  1.0,  0.0, -1.0,  2.5], dtype=tf.float32)

# MeanSquaredError
mse = tf.keras.losses.MeanSquaredError()
def mse_scratch(y_true, y_pred): return tf.reduce_mean((y_pred - y_true)**2)
check('MSE', mse_scratch(y_true, y_pred), mse(y_true, y_pred))

# MeanAbsoluteError
mae = tf.keras.losses.MeanAbsoluteError()
def mae_scratch(y_true, y_pred): return tf.reduce_mean(tf.abs(y_pred - y_true))
check('MAE', mae_scratch(y_true, y_pred), mae(y_true, y_pred))

# Huber
huber = tf.keras.losses.Huber(delta=1.0)
def huber_scratch(y_true, y_pred, delta=1.0):
    err = tf.abs(y_pred - y_true)
    return tf.reduce_mean(tf.where(err <= delta, 0.5*err**2, delta*(err - 0.5*delta)))
check('Huber', huber_scratch(y_true, y_pred), huber(y_true, y_pred))

print(f'\nPyTorch vs TensorFlow API:')
print('  nn.MSELoss()       ↔  tf.keras.losses.MeanSquaredError()')
print('  nn.L1Loss()        ↔  tf.keras.losses.MeanAbsoluteError()')
print('  nn.HuberLoss()     ↔  tf.keras.losses.Huber()')

In [ ]:
# Gradient via GradientTape
y_pred_var = tf.Variable([ 1.5, -0.5,  2.0,  0.0,  3.0], dtype=tf.float32)
y_true_const = tf.constant([ 1.0,  1.0,  0.0, -1.0,  2.5], dtype=tf.float32)

with tf.GradientTape() as tape:
    loss = mse(y_true_const, y_pred_var)
grad = tape.gradient(loss, y_pred_var)

analytical = 2 * (y_pred_var - y_true_const) / len(y_pred_var)
check('MSE gradient', grad, analytical)
print(f'Gradient: {grad.numpy()}')

## Section 2 — Binary Classification: BinaryCrossentropy

**Key:** Always use `from_logits=True` for numerical stability. This fuses sigmoid and BCE into a single stable operation.

In [ ]:
logits = tf.constant([-2.0, -1.0, 0.0, 1.0, 2.0])
probs  = tf.sigmoid(logits)
labels = tf.constant([0.0,  0.0, 1.0, 1.0, 1.0])

# from_logits=True (recommended)
bce_logits = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# from_logits=False (requires probabilities)
bce_probs = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# Manual from logits (sigmoid cross entropy with logits)
def bce_from_logits_scratch(y_true, logits):
    return tf.reduce_mean(
        tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=logits)
    )

check('BCE from logits', bce_from_logits_scratch(labels, logits), bce_logits(labels, logits))

# Equivalence check
check('BCE equiv (logits vs probs)', bce_logits(labels, logits), bce_probs(labels, probs))

# Demonstrate why from_logits=True is safer
large_logits = tf.constant([100.0, -100.0])
large_labels = tf.constant([1.0, 0.0])
print(f'\nLarge logit test:')
print(f'  from_logits=True:  {bce_logits(large_labels, large_logits).numpy():.6f}')
print(f'  from_logits=False: {bce_probs(large_labels, tf.sigmoid(large_logits)).numpy()}')
print('  → from_logits=False may output nan or inf for extreme logits')

## Section 3 — Multi-class Classification: SparseCategoricalCrossentropy, CategoricalCrossentropy

In [ ]:
logits = tf.constant([[1.0, 2.0, 0.5],
                       [0.0, 1.0, 3.0],
                       [2.0, 0.0, 1.0]])
targets_int  = tf.constant([1, 2, 0])           # integer indices
targets_onehot = tf.one_hot(targets_int, depth=3)  # one-hot

scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
cce  = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# Manual: -log(softmax(logits)[target])
def ce_scratch(logits, targets_int):
    log_softmax = logits - tf.reduce_logsumexp(logits, axis=-1, keepdims=True)
    return -tf.reduce_mean(
        tf.gather(log_softmax, targets_int[:, None], batch_dims=1)
    )

check('SparseCCE', ce_scratch(logits, targets_int), scce(targets_int, logits))
check('CCE vs SCCE', scce(targets_int, logits), cce(targets_onehot, logits))
print('SparseCategoricalCrossentropy == CategoricalCrossentropy (one-hot targets) ✓')

## Section 4 — KL Divergence and Soft Targets

**API difference:** PyTorch `KLDivLoss` expects **log-probabilities** as input. TensorFlow `KLDivergence` expects **probabilities**.

In [ ]:
# TF KLDivergence: KL(y_true || y_pred) = sum(y_true * log(y_true/y_pred))
# Input: PROBABILITIES (not log-probs)
p = tf.nn.softmax(tf.random.normal([4, 5]))  # true distribution
q = tf.nn.softmax(tf.random.normal([4, 5]))  # predicted distribution

kl = tf.keras.losses.KLDivergence()
kl_manual = tf.reduce_mean(tf.reduce_sum(p * tf.math.log(p / q), axis=-1))
check('KLDivergence', kl_manual, kl(p, q))

print('\nTF vs PyTorch KLDiv API difference:')
print('  PyTorch: input=log(Q), target=P   (log-probabilities)')
print('  TF:      y_pred=Q, y_true=P       (probabilities)')

# Knowledge distillation example (soft targets)
temperature = 4.0
teacher_logits = tf.constant([[3.0, 1.0, 0.5, -0.5, 2.0]])
student_logits = tf.constant([[2.0, 0.5, 0.2, -0.3, 1.5]])

soft_teacher = tf.nn.softmax(teacher_logits / temperature)
soft_student = tf.nn.softmax(student_logits / temperature)
distill_loss = kl(soft_teacher, soft_student) * temperature**2
print(f'\nKnowledge distillation loss (T={temperature}): {distill_loss.numpy():.4f}')

## Section 5 — Custom Loss: From-Scratch Huber

Subclass `tf.keras.losses.Loss` to create a custom differentiable loss that integrates with `model.compile`.

In [ ]:
class HuberLossCustom(tf.keras.losses.Loss):
    def __init__(self, delta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.delta = delta
    
    def call(self, y_true, y_pred):
        err = tf.abs(tf.cast(y_true, tf.float32) - tf.cast(y_pred, tf.float32))
        return tf.where(err <= self.delta,
                        0.5 * err**2,
                        self.delta * (err - 0.5 * self.delta))
    
    def get_config(self):
        return {'delta': self.delta, **super().get_config()}

y_true = tf.constant([1.0, 2.0, 3.0, 4.0])
y_pred = tf.constant([1.2, 1.8, 4.0, 3.5])

custom = HuberLossCustom(delta=1.0)
builtin = tf.keras.losses.Huber(delta=1.0)
check('Custom Huber', tf.reduce_mean(custom(y_true, y_pred)), builtin(y_true, y_pred))

# Verify gradients flow through custom loss
y_pred_var = tf.Variable([1.2, 1.8, 4.0, 3.5], dtype=tf.float32)
with tf.GradientTape() as tape:
    loss = tf.reduce_mean(custom(y_true, y_pred_var))
grad = tape.gradient(loss, y_pred_var)
print(f'Custom Huber gradient: {grad.numpy()}  (non-None = gradients flow ✓)')

# Use with model.compile
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss=HuberLossCustom(delta=1.0), metrics=['mae'])

X_demo = np.random.randn(100, 10).astype(np.float32)
y_demo = np.random.randn(100, 1).astype(np.float32)
history = model.fit(X_demo, y_demo, epochs=3, verbose=0)
print(f'Custom loss training OK — final loss: {history.history["loss"][-1]:.4f}')

## Section 6 — Custom Triplet Loss with GradientTape

In [ ]:
class TripletLoss(tf.keras.losses.Loss):
    def __init__(self, margin=1.0, **kwargs):
        super().__init__(**kwargs)
        self.margin = margin
    
    def call(self, anchor, positive_negative):
        # Expects anchor and a stacked tensor [positive, negative] or separate inputs
        positive = positive_negative[:, 0]
        negative = positive_negative[:, 1]
        d_ap = tf.norm(anchor - positive, axis=-1)
        d_an = tf.norm(anchor - negative, axis=-1)
        return tf.maximum(d_ap - d_an + self.margin, 0.0)

# Standalone triplet loss function
def triplet_loss_fn(anchor, positive, negative, margin=1.0):
    d_ap = tf.norm(anchor - positive, axis=-1)
    d_an = tf.norm(anchor - negative, axis=-1)
    return tf.reduce_mean(tf.maximum(d_ap - d_an + margin, 0.0))

# Verify gradients via GradientTape
tf.random.set_seed(0)
D = 16
anchor   = tf.Variable(tf.math.l2_normalize(tf.random.normal([8, D]), axis=1))
positive = tf.Variable(tf.math.l2_normalize(tf.random.normal([8, D]), axis=1))
negative = tf.Variable(tf.math.l2_normalize(tf.random.normal([8, D]), axis=1))

with tf.GradientTape() as tape:
    loss = triplet_loss_fn(anchor, positive, negative)

grads = tape.gradient(loss, [anchor, positive, negative])
print(f'Triplet loss: {loss.numpy():.4f}')
for name, g in zip(['anchor', 'positive', 'negative'], grads):
    print(f'  grad {name}: norm={tf.norm(g).numpy():.4f}, shape={g.shape}')

# One gradient step
optimizer = tf.keras.optimizers.Adam(1e-3)
optimizer.apply_gradients(zip(grads, [anchor, positive, negative]))
loss_after = triplet_loss_fn(anchor, positive, negative)
print(f'\nLoss before step: {loss.numpy():.4f}')
print(f'Loss after step:  {loss_after.numpy():.4f}  (should decrease)')

## Section 7 — Numerical Parity Check: TF vs PyTorch

In [ ]:
try:
    import torch
    import torch.nn as pt_nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not available in this environment. Skipping parity check.')

if TORCH_AVAILABLE:
    np.random.seed(42)
    y_pred_np = np.random.randn(8).astype(np.float32)
    y_true_np = np.random.randn(8).astype(np.float32)
    logits_np = np.random.randn(8, 4).astype(np.float32)
    targets_np = np.array([0, 1, 2, 3, 0, 1, 2, 3], dtype=np.int64)

    results = {}

    # MSE
    tf_mse  = tf.keras.losses.MeanSquaredError()(y_true_np, y_pred_np).numpy()
    pt_mse  = pt_nn.MSELoss()(torch.tensor(y_pred_np), torch.tensor(y_true_np)).item()
    results['MSE'] = abs(tf_mse - pt_mse)

    # MAE
    tf_mae  = tf.keras.losses.MeanAbsoluteError()(y_true_np, y_pred_np).numpy()
    pt_mae  = pt_nn.L1Loss()(torch.tensor(y_pred_np), torch.tensor(y_true_np)).item()
    results['MAE'] = abs(tf_mae - pt_mae)

    # Huber
    tf_hub  = tf.keras.losses.Huber(delta=1.0)(y_true_np, y_pred_np).numpy()
    pt_hub  = pt_nn.HuberLoss(delta=1.0)(torch.tensor(y_pred_np), torch.tensor(y_true_np)).item()
    results['Huber'] = abs(tf_hub - pt_hub)

    # SparseCCE vs CrossEntropy
    tf_cce  = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)(targets_np, logits_np).numpy()
    pt_cce  = pt_nn.CrossEntropyLoss()(torch.tensor(logits_np), torch.tensor(targets_np)).item()
    results['CrossEntropy'] = abs(tf_cce - pt_cce)

    print('Parity check: TF vs PyTorch')
    print(f'{"Loss":<20} {"Max Abs Diff":<15}')
    print('-' * 35)
    for name, diff in results.items():
        status = '✓' if diff < 1e-5 else '✗'
        print(f'{status} {name:<18} {diff:.2e}')

## Section 8 — End-to-End Training with model.compile

In [ ]:
# Regression: Huber loss
np.random.seed(42)
X_reg = np.random.randn(1000, 10).astype(np.float32)
y_reg = (X_reg[:, 0] * 2 + X_reg[:, 1] - X_reg[:, 2] * 0.5 + np.random.randn(1000) * 0.3).astype(np.float32)

reg_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
reg_model.compile(optimizer='adam', loss=tf.keras.losses.Huber(delta=1.0), metrics=['mae'])
reg_hist = reg_model.fit(X_reg, y_reg, epochs=20, validation_split=0.2, verbose=0)
print(f'Regression (Huber) — final val MAE: {reg_hist.history["val_mae"][-1]:.4f}')

In [ ]:
# Classification: SparseCategoricalCrossentropy on Fashion-MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

clf_model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10)  # No softmax — from_logits=True
])
clf_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
clf_hist = clf_model.fit(
    X_train, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=256,
    verbose=1
)
test_loss, test_acc = clf_model.evaluate(X_test, y_test, verbose=0)
print(f'\nFashion-MNIST test accuracy: {test_acc:.4f}')

In [ ]:
# Demo: sample_weight as TF equivalent of PyTorch reduction='none' + manual weighting
print('Sample weighting demo (TF equivalent of PyTorch reduction=none + weighting):')
y_true_w = tf.constant([0.0, 1.0, 1.0, 0.0, 1.0])
y_pred_w = tf.constant([-0.5, 1.2, -0.3, 0.8, 2.0])
# Higher weight for positive examples
weights  = tf.constant([1.0, 2.0, 2.0, 1.0, 2.0])

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
unweighted = bce(y_true_w, y_pred_w).numpy()
weighted   = bce(y_true_w, y_pred_w, sample_weight=weights).numpy()
print(f'  Unweighted loss: {unweighted:.4f}')
print(f'  Weighted loss:   {weighted:.4f}')

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(clf_hist.history['loss'], label='Train')
axes[0].plot(clf_hist.history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('SparseCategoricalCrossentropy'); axes[0].legend()
axes[1].plot(clf_hist.history['accuracy'], label='Train')
axes[1].plot(clf_hist.history['val_accuracy'], label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()